# YOLOv9 Instance Segmentation with LibreYOLO

This tutorial walks through the instance segmentation capability added to LibreYOLO's YOLOv9 family (branch `47-add-instance-segmentation-to-yolo9`).

You'll learn how to:
1. Build a seg model from scratch or load a detection checkpoint and attach a seg head.
2. Prepare a YOLO-polygon dataset.
3. Fine-tune on a small dataset.
4. Run inference and visualize masks.
5. Push trained weights to the Hugging Face Hub.

The end-to-end flow is deliberately small so the notebook completes on a CPU in a few minutes. The same code works unchanged on GPU/MPS and on full-scale datasets like COCO-seg.

---

## 1. Install LibreYOLO

From source (recommended while seg is on a feature branch):

```bash
git clone -b 47-add-instance-segmentation-to-yolo9 https://github.com/LibreYOLO/libreyolo
cd libreyolo
pip install -e .
```

Or from this fork (which includes an imgsz-agnostic fix for seg targets):

```bash
git clone -b agentic/a-yolo9-segmentation https://github.com/aalvsz/libreyolo
cd libreyolo
pip install -e .
```

In [ ]:
import numpy as np
import torch
import yaml
from pathlib import Path
from PIL import Image

from libreyolo.models.yolo9.model import LibreYOLO9
from libreyolo.models.yolo9.nn import LibreYOLO9Model

print('torch:', torch.__version__)
print('mps available:', torch.backends.mps.is_available())

## 2. Build a tiny YOLO-seg dataset

A YOLOv9 seg dataset is a standard YOLO directory layout where each label line can be either:

| format | columns |
|--------|---------|
| detection | `class cx cy w h` |
| segmentation | `class x1 y1 x2 y2 x3 y3 ...` (polygon with ≥ 3 vertices) |

All coordinates are normalized to `[0, 1]`. The dataset auto-detects which rows are polygons based on the number of columns.

Below we synthesize a trivial 4-image dataset: each image has one bright square at a random location, labeled with a 4-vertex polygon tracing its outline.

In [ ]:
DATASET = Path('demo_seg_dataset').resolve()
for split in ('train', 'val'):
    (DATASET / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET / 'labels' / split).mkdir(parents=True, exist_ok=True)

def make_sample(split, idx, imgsz=128, cls=0):
    rng = np.random.default_rng(idx)
    img = rng.integers(60, 110, size=(imgsz, imgsz, 3), dtype=np.uint8)
    x0, y0 = rng.integers(10, imgsz // 2, size=2)
    x1, y1 = x0 + rng.integers(30, imgsz - x0 - 4), y0 + rng.integers(30, imgsz - y0 - 4)
    img[y0:y1, x0:x1] = 220
    Image.fromarray(img).save(DATASET / 'images' / split / f'{idx}.jpg', quality=85)
    xs = [x0, x1, x1, x0]
    ys = [y0, y0, y1, y1]
    coords = []
    for x, y in zip(xs, ys):
        coords.extend([f'{x / imgsz:.6f}', f'{y / imgsz:.6f}'])
    (DATASET / 'labels' / split / f'{idx}.txt').write_text(f'{cls} ' + ' '.join(coords) + '\n')

for i in range(8):
    make_sample('train', i)
for i in range(2):
    make_sample('val', i)

data_yaml = DATASET / 'data.yaml'
data_yaml.write_text(yaml.dump({
    'path': str(DATASET),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['square'],
}))
print(f'Dataset ready: {DATASET}')

## 3. Instantiate a seg model

Three common starting points:

1. **From scratch** — build `LibreYOLO9Model(segmentation=True)` and save its state dict. Use this when you want full control over initialization.
2. **From a detection checkpoint** — pass `LibreYOLO9('LibreYOLO9t.pt', size='t', segmentation=True)`. The backbone/neck weights load and the new seg head is initialized randomly. This is the standard detection→segmentation fine-tuning flow.
3. **From a seg checkpoint** — pass a `.pt` file whose state dict already contains `head.proto` / `head.cv4` keys. `segmentation=True` is auto-detected.

For a self-contained notebook that runs on CPU without network access, we use approach 1.

In [ ]:
init_ckpt = DATASET / 'yolo9t-seg-init.pt'
net = LibreYOLO9Model(config='t', nb_classes=1, segmentation=True)
torch.save({'model': net.state_dict()}, init_ckpt)

model = LibreYOLO9(
    model_path=str(init_ckpt),
    size='t',
    nb_classes=1,
    segmentation=True,
    device='cpu',
)
assert model._is_segmentation
print('Seg model ready. Params:', sum(p.numel() for p in model.model.parameters()))

## 4. Train briefly

Real COCO-seg fine-tuning needs `imgsz=640`, `batch=16+`, and many epochs. We keep it tiny here so the cell finishes on CPU. For a production training run, use `scripts/train_yolo9_seg.py` — see the blog post for HF Jobs submission.

In [ ]:
results = model.train(
    data=str(data_yaml),
    epochs=2,
    batch=2,
    imgsz=128,
    lr0=0.01,
    optimizer='SGD',
    device='cpu',
    workers=0,
    project=str(DATASET / 'runs'),
    name='demo',
    exist_ok=True,
    amp=False,
    patience=2,
)
print('Final loss:', results['final_loss'])
print('Best checkpoint:', results['best_checkpoint'])

## 5. Run inference

`LibreYOLO9(...)` exposes `__call__` for inference. The returned `Results` object has:

- `.boxes` — `Boxes` with `.xyxy`, `.conf`, `.cls`
- `.masks` — `Masks` with `.data` (N, H, W binary), `.xy` (pixel contours), `.xyn` (normalized contours)
- `.orig_shape` — (H, W) of the input image
- `.names` — class id → name map

In [ ]:
best = LibreYOLO9(model_path=results['best_checkpoint'], size='t', nb_classes=1, segmentation=True, device='cpu')
sample = DATASET / 'images' / 'val' / '0.jpg'
result = best(sample, conf=0.001, iou=0.5)
res = result[0] if isinstance(result, list) else result
print('detections:', len(res.boxes))
if res.masks is not None:
    print('masks tensor shape:', tuple(res.masks.data.shape))

## 6. Visualize masks

Even with 2 epochs on 8 images, masks will be noisy — this is a plumbing demo, not a benchmark. A real training run (COCO-seg, 100+ epochs) produces tight masks.

In [ ]:
import matplotlib.pyplot as plt

img = np.array(Image.open(sample))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img); axes[0].set_title('input'); axes[0].axis('off')

overlay = img.astype(np.float32)
if res.masks is not None and res.masks.data.shape[0] > 0:
    m = res.masks.data[0].cpu().numpy().astype(bool)
    overlay[m] = overlay[m] * 0.5 + np.array([255, 60, 60]) * 0.5
axes[1].imshow(overlay.astype(np.uint8)); axes[1].set_title('prediction (masked)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 7. Push to HF Hub

Once you've trained a real checkpoint (e.g. on COCO-seg), publish it with:

```bash
python scripts/train_yolo9_seg.py \
    --data coco-seg/data.yaml \
    --size s --epochs 300 --batch 16 --imgsz 640 \
    --push --hf-repo ander2221/libreyolo-yolo9s-seg
```

The script uploads `best.pt` and a minimal model card. Reload with:

```python
from huggingface_hub import hf_hub_download
from libreyolo import LibreYOLO9

ckpt = hf_hub_download(repo_id='ander2221/libreyolo-yolo9s-seg', filename='best.pt')
model = LibreYOLO9(model_path=ckpt, size='s', segmentation=True)
```

See `docs/agentic-features/blog/yolo9-instance-segmentation.md` for the full story.